In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

from src.data_loader import load_data, basic_cleaning
from src.hypothesis_tests import claim_frequency_test, numerical_ttest

# Task 3: A/B Hypothesis Testing

The objective of this notebook is to statistically test whether insurance risk and profitability differ across customer, geographic, and demographic segments.

The analysis supports ACIS in making evidence-based pricing and marketing decisions.

In [ ]:
DATA_PATH = "../data/insurance_data_cleaned.csv"

df = load_data(DATA_PATH)
df = basic_cleaning(df)

df.head()

## KPI Definitions

- **Claim Frequency**: proportion of policies with at least one claim.
- **Claim Severity**: average claim amount among policies with claims.
- **Margin**: TotalPremium − TotalClaims.
- **Loss Ratio**: TotalClaims / TotalPremium.

In [ ]:
df.columns.tolist()

In [ ]:
df["Province"].value_counts().head()

In [ ]:
province_groups = df["Province"].value_counts().head(2).index.tolist()

province_a = province_groups[0]
province_b = province_groups[1]

province_test = claim_frequency_test(
    df=df,
    group_col="Province",
    group_a=province_a,
    group_b=province_b
)

province_test

In [ ]:
postal_groups = df["PostalCode"].value_counts().head(2).index.tolist()

postal_a = postal_groups[0]
postal_b = postal_groups[1]

postal_risk_test = claim_frequency_test(
    df=df,
    group_col="PostalCode",
    group_a=postal_a,
    group_b=postal_b
)

postal_risk_test

In [ ]:
postal_margin_test = numerical_ttest(
    df=df,
    group_col="PostalCode",
    group_a=postal_a,
    group_b=postal_b,
    target_col="Margin"
)

postal_margin_test

In [ ]:
df["Gender"].value_counts()

In [ ]:
gender_values = df["Gender"].dropna().unique().tolist()
gender_values

In [ ]:
gender_test = claim_frequency_test(
    df=df,
    group_col="Gender",
    group_a="Male",
    group_b="Female"
)

gender_test

In [ ]:
results = pd.DataFrame([
    {
        "Hypothesis": "No risk difference across provinces",
        "KPI": "Claim Frequency",
        "Test Used": province_test["test"],
        "Groups Compared": f"{province_test['group_a']} vs {province_test['group_b']}",
        "P-value": province_test["p_value"],
        "Decision": province_test["decision"]
    },
    {
        "Hypothesis": "No risk difference between postal codes",
        "KPI": "Claim Frequency",
        "Test Used": postal_risk_test["test"],
        "Groups Compared": f"{postal_risk_test['group_a']} vs {postal_risk_test['group_b']}",
        "P-value": postal_risk_test["p_value"],
        "Decision": postal_risk_test["decision"]
    },
    {
        "Hypothesis": "No margin difference between postal codes",
        "KPI": "Margin",
        "Test Used": postal_margin_test["test"],
        "Groups Compared": f"{postal_margin_test['group_a']} vs {postal_margin_test['group_b']}",
        "P-value": postal_margin_test["p_value"],
        "Decision": postal_margin_test["decision"]
    },
    {
        "Hypothesis": "No risk difference between women and men",
        "KPI": "Claim Frequency",
        "Test Used": gender_test["test"],
        "Groups Compared": f"{gender_test['group_a']} vs {gender_test['group_b']}",
        "P-value": gender_test["p_value"],
        "Decision": gender_test["decision"]
    }
])

results

# Business Interpretation and Recommendations

The hypothesis testing results provide evidence for whether ACIS should adjust pricing or marketing strategies across geographic and demographic segments.

## Recommendations

- If province-level risk differences are statistically significant, ACIS should consider province-specific premium adjustments.
- If postal-code-level claim frequency differs significantly, postal code should be considered in risk segmentation.
- If margin differs significantly between postal codes, ACIS can prioritize profitable postal codes for marketing campaigns.
- If gender-based risk differences are not statistically significant, pricing decisions should avoid gender-based adjustments and instead rely on stronger risk indicators such as geography, vehicle type, and claims history.

These findings should be combined with model-based risk predictions before making final pricing decisions.